# Train a wake word for Luni (microWakeWord -> ESP32-S3)

Reusable, self-contained recipe. Produces a quantized INT8 streaming TFLite + a ready-to-use
`model_<slug>.h` C-array for the firmware (`esp32-s3/lib/wakeword/`).

**How to use (new word = edit CONFIG, then Run all)**
1. Upload this notebook to https://colab.research.google.com
2. **Runtime -> Change runtime type -> T4 GPU** (free). Run-all stops early with a clear message if no GPU.
3. Edit the CONFIG cell (`WAKE_PHRASE`, `MODEL_SLUG`) — those two are the only knobs for a new word.
4. **Runtime -> Run all** (~25-35 min). It auto-downloads `model_<slug>.h` (+ the `.tflite`) at the end.
5. Copy `model_<slug>.h` into `esp32-s3/lib/wakeword/`, point the firmware at it
   (`WakeWordDetector.cpp`: `#include` + `GetModel(g_<slug>_model)`), and set `DETECT_THRESHOLD` in
   `esp32-s3/lib/wakeword/WakeWordConfig.hpp` using the **RECOMMENDED** value printed in step 7.

This bakes in the fixes needed as of 2026-06 (TF 2.20 + datasets 4.x on Colab). See `docs/WAKEWORD.md`.
Current embedded word: **"Hey Luni"** (`model_hey_luni.h`, `DETECT_THRESHOLD=0.75`).

## 0. Config + GPU check

In [ ]:
# ====== EDIT ME (then Runtime > Run all) ======
WAKE_PHRASE    = "hey loonie"  # how Piper should SAY it; use a phonetic spelling ("hey loonie" => "loo-nee"). Listen at step 2.
MODEL_SLUG     = "hey_luni"    # output = model_<slug>.h ; C symbol g_<slug>_model  (a-z 0-9 _ only, start with a letter)
NUM_POSITIVES  = 2000          # synthetic TTS samples to generate
TRAINING_STEPS = 10000         # T4 ~20-30 min
# ===============================================
import shutil, subprocess, re
assert re.fullmatch(r"[a-z][a-z0-9_]*", MODEL_SLUG), "MODEL_SLUG must be a valid C identifier: a-z 0-9 _ , starting with a letter"
if shutil.which("nvidia-smi") is None:
    raise SystemExit("No GPU detected. Set Runtime > Change runtime type > T4 GPU, then Run all again.")
print("GPU:", subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True).stdout.strip())
print("Wake phrase: %r  ->  model_%s.h (symbol g_%s_model)" % (WAKE_PHRASE, MODEL_SLUG, MODEL_SLUG))

## 1. Install microWakeWord + Piper (with the 2026 fixes)

In [ ]:
import os
# audio-metadata fork (unpins attrs) — used by microWakeWord
%pip -q install "git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"
if not os.path.exists("/content/microWakeWord"):
    !git clone -q https://github.com/kahrendt/microWakeWord /content/microWakeWord
%pip -q install -e /content/microWakeWord
# piper-tts for sample generation; datasets<4 (4.x audio API breaks microWakeWord's Clips)
%pip -q install piper-tts==1.3.0 "datasets<4.0" soundfile librosa
if not os.path.exists("/content/piper-sample-generator"):
    !git clone -q https://github.com/rhasspy/piper-sample-generator /content/piper-sample-generator
MODELPT = "/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"
if not os.path.exists(MODELPT):
    !wget -q -O {MODELPT} https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
print("install done")

In [ ]:
# Patch microWakeWord for TensorFlow 2.20 (metric results are ndarrays now, code calls .numpy())
import re
def _patch(path, subs):
    s = open(path).read()
    if not re.search(r"^import numpy as np$", s, re.M):
        s = re.sub(r"(^import .*$)", r"import numpy as np\n\1", s, count=1, flags=re.M)
    for a,b in subs: s = re.sub(a,b,s)
    open(path,"w").write(s)
_patch("/content/microWakeWord/microwakeword/train.py", [(r'(\w+\["[^"]+"\])\.numpy\(\)', r'np.asarray(\1)')])
_patch("/content/microWakeWord/microwakeword/test.py",  [(r'\bresult\.numpy\(\)', r'np.asarray(result)')])
print("patched train.py + test.py")

## 2. Generate positive samples (Piper TTS)\nListen to the single sample below; if the pronunciation is wrong, set `WAKE_PHRASE` to a phonetic spelling and re-run.

In [ ]:
import subprocess
REPO="/content/piper-sample-generator"; M="models/en_US-libritts_r-medium.pt"
subprocess.run(["python","-m","piper_sample_generator",WAKE_PHRASE,"--model",M,
    "--max-samples","1","--batch-size","1","--max-speakers","800","--output-dir","/content/verify"],cwd=REPO)
from IPython.display import Audio
Audio("/content/verify/0.wav")

In [ ]:
import subprocess, glob, os, time
REPO="/content/piper-sample-generator"; M="models/en_US-libritts_r-medium.pt"
os.system("rm -rf /content/generated_samples"); t=time.time()
subprocess.run(["python","-m","piper_sample_generator",WAKE_PHRASE,"--model",M,
    "--max-samples",str(NUM_POSITIVES),"--batch-size","128","--max-speakers","800",
    "--length-scales","0.75","1.0","1.25","--output-dir","/content/generated_samples"],cwd=REPO)
print(len(glob.glob("/content/generated_samples/*.wav")),"positives in %.0fs"%(time.time()-t))

## 3. Backgrounds (RIR + ESC-50) + negative feature datasets

In [ ]:
import os, numpy as np, scipy.io.wavfile, datasets
from pathlib import Path
print("datasets", datasets.__version__, "(must be 3.x)")
# MIT impulse responses (reverb)
if not os.path.exists("/content/mit_rirs"):
    os.makedirs("/content/mit_rirs")
    for r in datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses",split="train",streaming=True):
        scipy.io.wavfile.write("/content/mit_rirs/"+r["audio"]["path"].split("/")[-1],16000,(r["audio"]["array"]*32767).astype(np.int16))
# ESC-50 environmental noise -> 16k (reliable; AudioSet/FMA from the upstream notebook are gated/broken)
if not os.path.exists("/content/esc50_16k"):
    if not os.path.exists("/content/esc50.zip"):
        !wget -q -O /content/esc50.zip https://github.com/karolpiczak/ESC-50/archive/master.zip
    !unzip -q -o /content/esc50.zip -d /content/esc50_src
    os.makedirs("/content/esc50_16k")
    files=[str(p) for p in Path("/content/esc50_src").glob("**/audio/*.wav")]
    d=datasets.Dataset.from_dict({"audio":files}).cast_column("audio",datasets.Audio(sampling_rate=16000))
    for r in d:
        scipy.io.wavfile.write("/content/esc50_16k/"+r["audio"]["path"].split("/")[-1],16000,(r["audio"]["array"]*32767).astype(np.int16))
# precomputed negative spectrogram datasets
if not os.path.exists("/content/negative_datasets"):
    os.makedirs("/content/negative_datasets")
    root="https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
    for f in ["dinner_party.zip","dinner_party_eval.zip","no_speech.zip","speech.zip"]:
        !wget -q -O /content/negative_datasets/{f} {root}{f}
        !unzip -q -o /content/negative_datasets/{f} -d /content/negative_datasets
print("RIR",len(os.listdir("/content/mit_rirs")),"| ESC50",len(os.listdir("/content/esc50_16k")))

## 4. Generate spectrogram features (40ch / 16k / 10ms)

In [ ]:
import sys, os; sys.path.insert(0,"/content/microWakeWord")
from microwakeword.audio.clips import Clips
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
clips=Clips(input_directory="/content/generated_samples",file_pattern="*.wav",max_clip_duration_s=None,
            remove_silence=False,random_split_seed=10,split_count=0.1)
aug=Augmentation(augmentation_duration_s=3.2,
    augmentation_probabilities={"SevenBandParametricEQ":0.1,"TanhDistortion":0.1,"PitchShift":0.1,
        "BandStopFilter":0.1,"AddColorNoise":0.1,"AddBackgroundNoise":0.75,"Gain":1.0,"RIR":0.5},
    impulse_paths=["/content/mit_rirs"],background_paths=["/content/esc50_16k"],
    background_min_snr_db=-5,background_max_snr_db=10,min_jitter_s=0.195,max_jitter_s=0.205)
os.makedirs("/content/features",exist_ok=True)
for split,(nm,rep,sl) in {"training":("train",2,10),"validation":("validation",1,10),"testing":("test",1,1)}.items():
    od="/content/features/"+split; os.makedirs(od,exist_ok=True)
    sg=SpectrogramGeneration(clips=clips,augmenter=aug,slide_frames=sl,step_ms=10)
    RaggedMmap.from_generator(out_dir=od+"/wakeword_mmap",
        sample_generator=sg.spectrogram_generator(split=nm,repeat=rep),batch_size=100,verbose=True)
print("features done")

## 5. Trim the validation-ambient set (avoids an OOM in training-time eval on Colab's ~13GB RAM)

In [ ]:
from mmap_ninja.ragged import RaggedMmap
import numpy as np, os, shutil
def trim(d, keep, bak="/content/ambient_backup"):
    os.makedirs(bak,exist_ok=True); b=os.path.join(bak,os.path.basename(d))
    if os.path.exists(b): print("already trimmed"); return
    m=RaggedMmap(d); items=[np.asarray(m[i]) for i in range(min(keep,len(m)))]
    shutil.move(d,b)
    RaggedMmap.from_generator(out_dir=d,sample_generator=(x for x in items),batch_size=2,verbose=False)
    print("trimmed",os.path.basename(d),"->",len(RaggedMmap(d)))
# only validation_ambient (used during training); testing_ambient stays full for the eval in step 7
trim("/content/negative_datasets/dinner_party_eval/validation_ambient/chime6_dev_eval_mmap", 1)

## 6. Train (T4 ~20-30 min). The built-in faph table may print `nan` (a version quirk) — step 7 computes the real one.

In [ ]:
import yaml
cfg={"window_step_ms":10,"train_dir":"/content/trained","features":[
 {"features_dir":"/content/features","sampling_weight":2.0,"penalty_weight":1.0,"truth":True,"truncation_strategy":"truncate_start","type":"mmap"},
 {"features_dir":"/content/negative_datasets/speech","sampling_weight":10.0,"penalty_weight":1.0,"truth":False,"truncation_strategy":"random","type":"mmap"},
 {"features_dir":"/content/negative_datasets/dinner_party","sampling_weight":10.0,"penalty_weight":1.0,"truth":False,"truncation_strategy":"random","type":"mmap"},
 {"features_dir":"/content/negative_datasets/no_speech","sampling_weight":5.0,"penalty_weight":1.0,"truth":False,"truncation_strategy":"random","type":"mmap"},
 {"features_dir":"/content/negative_datasets/dinner_party_eval","sampling_weight":0.0,"penalty_weight":1.0,"truth":False,"truncation_strategy":"split","type":"mmap"}],
 "training_steps":[TRAINING_STEPS],"positive_class_weight":[1],"negative_class_weight":[20],"learning_rates":[0.001],
 "batch_size":128,"time_mask_max_size":[0],"time_mask_count":[0],"freq_mask_max_size":[0],"freq_mask_count":[0],
 "eval_step_interval":500,"clip_duration_ms":1500,"target_minimization":0.9,"minimization_metric":None,
 "maximization_metric":"average_viable_recall"}
yaml.dump(cfg,open("/content/train.yaml","w")); print("yaml written")

In [ ]:
!python -m microwakeword.model_train_eval --training_config=/content/train.yaml \
 --train 1 --restore_checkpoint 0 --test_tflite_streaming_quantized 1 --use_weights "best_weights" \
 mixednet --pointwise_filters "64,64,64,64" --repeat_in_block "1, 1, 1, 1" \
 --mixconv_kernel_sizes "[5], [7,11], [9,15], [23]" --residual_connection "0,0,0,0" \
 --first_conv_filters 32 --first_conv_kernel_size 5 --stride 3

## 7. Threshold table (recall vs false-accepts/hour) — pick `DETECT_THRESHOLD` from this

In [ ]:
import numpy as np, sys; sys.path.insert(0,"/content/microWakeWord")
from microwakeword.test import Model
from mmap_ninja.ragged import RaggedMmap
TFL="/content/trained/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
m=Model(TFL,stride=3); cut=np.round(np.arange(0.5,1.0,0.05),2); SW=5
def sp(t):
    p=np.asarray(m.predict_spectrogram(np.asarray(t).astype(np.float32))).flatten()
    return np.convolve(p,np.ones(SW)/SW,"valid") if len(p)>=SW else p
amb=RaggedMmap("/content/negative_datasets/dinner_party_eval/testing_ambient/dipco_u01_ch1_mmap")
fa=np.zeros(len(cut)); dur=0.0
for i in range(len(amb)):
    p=sp(amb[i]); dur+=len(p)*3*0.01/3600.0; cd=np.ones(len(cut))*25
    for x in p:
        cd=np.maximum(cd-1,0); det=x>cut
        for j in range(len(cut)):
            if cd[j]==0 and det[j]: fa[j]+=1; cd[j]=25
pos=RaggedMmap("/content/features/testing/wakeword_mmap")
mx=np.array([(sp(pos[i]).max() if len(sp(pos[i])) else 0.0) for i in range(len(pos))])
recall=np.array([(mx>c).mean() for c in cut]); faph=np.array([fa[j]/dur if dur else np.nan for j in range(len(cut))])
print("ambient %.0f min | %d positive clips\n"%(dur*60,len(pos)))
print("cutoff | recall | FA/hour")
for j,c in enumerate(cut): print("  %.2f  | %.3f  |  %.2f"%(c,recall[j],faph[j]))
# Auto-pick: highest recall, tie-break by lowest false-accepts/hour
best=np.lexsort((faph, -recall))[0]
print("\n>>> RECOMMENDED DETECT_THRESHOLD = %.2ff   (recall %.3f, %.2f false-accepts/hour)"%(cut[best],recall[best],faph[best]))
print(">>> Set it in esp32-s3/lib/wakeword/WakeWordConfig.hpp . Raise toward 0.90 if real-world false triggers; lower if it misses.")

## 8. Export `model_<slug>.h` + download (copy the .h into `esp32-s3/lib/wakeword/`)

In [ ]:
import base64, hashlib
TFL="/content/trained/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
raw=open(TFL,"rb").read()
print("size",len(raw),"bytes | sha256",hashlib.sha256(raw).hexdigest())
body="\n".join("  "+", ".join("0x%02x"%b for b in raw[i:i+12])+"," for i in range(0,len(raw),12))
h=('#pragma once\n'
   '// "%s" wake-word model (microWakeWord, quantized INT8 streaming TFLite)\n'
   '// 16 kHz, 40-ch spectrogram, 30 ms window, 10 ms stride. size=%d bytes\n'
   '#include <cstdint>\n'
   'alignas(16) const unsigned char g_%s_model[] = {\n%s\n};\n'
   'const unsigned int g_%s_model_len = %d;\n') % (WAKE_PHRASE, len(raw), MODEL_SLUG, body, MODEL_SLUG, len(raw))
open("/content/model_%s.h"%MODEL_SLUG,"w").write(h)
from google.colab import files
files.download(TFL)
files.download("/content/model_%s.h"%MODEL_SLUG)
print("Downloaded model_%s.h -> copy into esp32-s3/lib/wakeword/  (symbol g_%s_model)"%(MODEL_SLUG,MODEL_SLUG))